*This is the notebook for the inference phase of this example*

## Recurrent Neural Network Demonstration
**Goal:** Predict the function $f(x + \frac{\pi}{50})$ from $f(x)$

**Strategy:**
The training strategy was all in the file `model.ipynb`, the inference process is much simpler.
1. Architecture: 1 explicit state + 3 hidden state
2. The explicit state will be at slot 0, hidden state will be at slot [1, 2, 3]
3. Now we test for 200 different starting x, each going forward 1000 times.
4. We load up our training result from a `.cache` file.


This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [1]:
import random
import math
import lktorch as lk
import os

CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")

### 2. Generate and preprocess

In [2]:
private_test = []
for i in range(0, 200):
    current_test = []
    for j in range(0, 1000):
        current_test.append(math.sin(math.pi * (i + j * 0.5) / 50))
    private_test.append(current_test)

print("Done fetching data!")

Done fetching data!


### 3. Helper Functions

In [3]:
def join_info(public_state, hidden_state):
    sz = len(public_state.dimension())
    perm = [sz-1]
    for i in range(sz-1): 
        perm.append(i)
    public_state = public_state.PermuteDimension(perm)
    hidden_state = hidden_state.PermuteDimension(perm)
    state = lk.Merge(public_state, hidden_state)

    invperm = []
    for i in range(sz - 1): 
        invperm.append(i + 1)
    invperm.append(0)
    return state.PermuteDimension(invperm)

def extract_public(tensor):
    return tensor.Slice([0], [0])

def extract_hidden(tensor):
    return tensor.Slice([1], [3])

def calculate_loss(model, LossFunction):
    input_data = lk.Tensor(private_test[0])
    dim = input_data.dimension()
    input_data = input_data.Reshape(dim + [1])
    input_data = join_info(input_data, lk.Zeros(dim + [3]))

    cur_tensor = input_data
    cum_loss = lk.Zeros([])
    for it in range(1, len(private_test)):
        cur_tensor = model(cur_tensor)
        public_state = extract_public(cur_tensor)
        hidden_state = extract_hidden(cur_tensor)
        
        test_state = lk.Tensor(private_test[it]).Reshape(dim + [1])
        cum_loss = cum_loss + lk.MeanAll(LossFunction(public_state, test_state))
        cur_tensor = join_info(test_state, hidden_state)
    cum_loss = cum_loss + lk.SumAll(cur_tensor) * 0
    return cum_loss / float(len(private_test))

### 4. Setting up the model

In [4]:
model = lk.Sequential([
    lk.LinearLayer(4, 8),
    lk.Tanh_Layer(),
    lk.LinearLayer(8, 8),
    lk.Tanh_Layer(),
    lk.LinearLayer(8, 4),
    lk.Tanh_Layer(),
    lk.ScalarMultiply_Layer(float(1.2))
])

lk.deactivate_backprop()

### 5. Measuring the untrained network

In [5]:

mse_loss = calculate_loss(model, lk.MSELoss)
mae_loss = calculate_loss(model, lk.MAELoss)
rmse_loss = calculate_loss(model, lk.RMSELoss)
print("No Training:")
print("MSE Loss: ", mse_loss)
print("MAE Loss: ", mae_loss)
print("RMSE Loss: ", rmse_loss)

No Training:
MSE Loss:  1.914
MAE Loss:  1.21354
RMSE Loss:  1.21354


### 6. Measuring the trained network

In [7]:

model.load_state_dict(lk.ReadFile(WEIGHT_PATH))

mse_loss = calculate_loss(model, lk.MSELoss)
mae_loss = calculate_loss(model, lk.MAELoss)
rmse_loss = calculate_loss(model, lk.RMSELoss)
print("Yes Training:")
print("MSE Loss: ", mse_loss)
print("MAE Loss: ", mae_loss)
print("RMSE Loss: ", rmse_loss)

Yes Training:
MSE Loss:  0.00284772
MAE Loss:  0.0269785
RMSE Loss:  0.0269786


As you can see, the network is extremely accurate!